# Fase 1 — Validação da janela temporal (2023–2025)

Escolhemos os anos de 2023 a 2025 porque são depois da troca de sistema de
registro da SSP (R.D.O. → S.P.J., que aconteceu entre 2022 e 2023) e depois
da pandemia. Neste notebook testamos se essa escolha se sustenta nos dados e
geramos a **Figura 1**.

Regra que definimos antes de olhar o gráfico: se a série mensal tiver um
degrau na passagem de 2022 para 2023, a troca de sistema ainda estava
acontecendo e a janela passa a ser 2024–2025. Se a série for contínua, a
janela 2023–2025 fica confirmada.

Os dados vêm de `data/processed/ssp_painel.csv`, que já está agregado por
mês. Não precisamos reler os arquivos originais.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from config import SSP_PAINEL_CSV, GRUPOS_NATUREZA
from estilo import aplicar_estilo, salvar
from figuras import plot_serie_mensal
from parse_ssp import normaliza

aplicar_estilo()
print("Python:", sys.executable)   # conferir se é o Python que tem as bibliotecas

painel = pd.read_csv(SSP_PAINEL_CSV, dtype={"codigo_ibge": str})
print(painel.shape, "| anos:", sorted(painel["ano"].unique()))

## 1. A série está completa?

Se faltar algum mês, o gráfico vai mostrar um degrau que não existe. Por
isso conferimos se cada ano tem 12 meses e quantas ocorrências ficaram sem
mês (`mes = 0`).

In [ ]:
sem_mes = painel.loc[painel["mes"] == 0, "ocorrencias"].sum()
print(f"ocorrências sem mês: {sem_mes} de {painel['ocorrencias'].sum():,}")

meses_por_ano = painel[painel["mes"] > 0].groupby("ano")["mes"].nunique()
print(meses_por_ano.to_string())
assert (meses_por_ano == 12).all(), "algum ano não tem os 12 meses"

## 2. Figura 1 — total mensal de ocorrências, 2022 a 2025

O eixo vertical começa em zero de propósito. Se cortássemos o eixo, qualquer
variação pequena ia parecer um degrau.

In [ ]:
fig = plot_serie_mensal(painel)
salvar(fig, "figura1_serie_mensal")

## 3. A mesma série, por natureza

O total pode esconder uma queda em furto compensada por uma alta em roubo.
Aqui cada natureza aparece na sua própria escala. O que importa é o formato
da linha, não o nível.

In [ ]:
fig = plot_serie_mensal(painel, GRUPOS_NATUREZA)
salvar(fig, "figura1b_serie_por_natureza")

## 4. Teste com números

Sazonalidade também pode parecer degrau. Por isso comparamos a variação de
cada natureza de um ano para o outro. A pergunta é: a variação de 2022 para
2023 é maior que a das outras viradas de ano? E ela vai na mesma direção em
todas as naturezas? Uma troca de sistema de registro faria as duas coisas.

In [ ]:
mapa = {normaliza(nat): g for g, lista in GRUPOS_NATUREZA.items() for nat in lista}
bloco = painel.assign(grupo=painel["natureza"].map(mapa)).dropna(subset=["grupo"])

anual = bloco.groupby(["grupo", "ano"])["ocorrencias"].sum().unstack("ano")
variacao = (anual.pct_change(axis=1) * 100).round(1).iloc[:, 1:]
variacao.columns = [f"{a - 1}→{a}" for a in anual.columns[1:]]
variacao

In [ ]:
primeira = variacao.iloc[:, 0]          # 2022→2023
demais = variacao.iloc[:, 1:]           # as outras viradas

print(f"2022→2023: {(primeira > 0).sum()} naturezas sobem, "
      f"{(primeira < 0).sum()} descem")
print(f"variação mediana (em módulo): 2022→2023 = {primeira.abs().median():.1f}% | "
      f"demais viradas = {demais.abs().stack().median():.1f}%")

fora = primeira.abs() > 2 * demais.abs().median(axis=1)
print("naturezas em que 2022→2023 passa do dobro do normal:",
      ", ".join(fora[fora].index) or "nenhuma")

## 5. Conclusão: janela 2023–2025 confirmada

- A série total não tem degrau em janeiro de 2023.
- Na virada 2022→2023, metade das naturezas sobe e metade desce. Uma troca
  de sistema faria quase todas irem para o mesmo lado.
- A variação mediana de 2022→2023 é parecida com a das outras viradas.
- As duas naturezas que fogem do padrão são `cvli` e `estupro_total`, que
  são justamente as que menos dependem do sistema de registro. Uma mudança
  de software apareceria no furto e no roubo, não nelas.

`ANOS_JANELA` continua `[2023, 2024, 2025]` no `config.py`. O ano de 2022
fica no painel só para comparação e não entra na base final.